# Domain 2: Tool Design & MCP Integration
## TS 2.2: Implement Structured Error Responses for MCP Tools
**Certification Focus**: CCAR-F Domain 2 (18% Exam Weight) | **Scenario Alignment**: Scenario 1 & 4

---

### 🎯 Key Architectural Principles
1. **The MCP `isError: True` Pattern**: Tool errors should set `isError: true` within the tool result payload rather than throwing uncaught HTTP 500 exceptions.
2. **Error Taxonomy**:
   - `"transient"`: Network timeouts, temporary rate limits -> `isRetryable: true`
   - `"validation"`: Malformed SKU, invalid format -> `isRetryable: false`
   - `"business"`: Policy limit violations (e.g. refund > $500) -> `isRetryable: false`, with user-friendly guidance
   - `"permission"`: Insufficient role/privileges -> `isRetryable: false`
3. **Valid Empty Results vs Access Failures**: A database query that successfully finds 0 matching rows is **not an error** (`isError: false, stock_count: 0, is_empty_result: true`). Returning errors for empty results causes the agent to misdiagnose system health.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.2 MCP Structured Errors.")

### 🚨 Step 2: Structured Error Helper & Tool Implementations
We implement `create_mcp_error` and tools demonstrating transient, validation, business, and empty result states.

In [ ]:
# Step 2: MCP Error Helper & Tools
def create_mcp_error(category: str, message: str, is_retryable: bool, details: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Formats an MCP structured error response conforming to certification standards."""
    return {
        "isError": True,
        "errorCategory": category,  # 'transient' | 'validation' | 'business' | 'permission'
        "isRetryable": is_retryable,
        "message": message,
        "details": details or {},
    }

def query_inventory(sku: str, simulate_mode: Optional[str] = None) -> Dict[str, Any]:
    """Simulates warehouse inventory lookups with structured error handling."""
    if simulate_mode == "timeout":
        return create_mcp_error("transient", "Warehouse database connection timed out after 3000ms.", is_retryable=True)
    if not sku.startswith("SKU-"):
        return create_mcp_error("validation", f"Invalid SKU format '{sku}'. Expected format: SKU-XXXX.", is_retryable=False)
    if sku == "SKU-99999":
        # Valid empty result: NOT an error!
        return {"isError": False, "sku": sku, "stock_count": 0, "is_empty_result": True}
    return {"isError": False, "sku": sku, "stock_count": 42, "is_empty_result": False}

def issue_refund(customer_id: str, amount: float) -> Dict[str, Any]:
    """Simulates refund processing with business policy bounds."""
    if amount > 500.0:
        return create_mcp_error(
            "business",
            f"Refund amount ${amount:.2f} exceeds automated policy threshold ($500.00). Escalate to manager.",
            is_retryable=False,
            details={"customer_id": customer_id, "threshold": 500.0, "requested": amount}
        )
    return {"isError": False, "customer_id": customer_id, "refund_amount": amount, "status": "success"}

print(" Structured error engine ready.")

### 🧪 Step 3: Interactive Simulation
We test all 4 primary error/result categories:
1. **Transient Timeout**: Retryable.
2. **Malformed SKU**: Non-retryable validation error.
3. **Valid Empty Result**: Success with 0 items (isError: False).
4. **Policy Cap Breach**: Business rule violation.

In [ ]:
# Step 3: Simulation
# 1. Transient Error
res_transient = query_inventory("SKU-101", simulate_mode="timeout")
print("Transient Error Result:", res_transient)

# 2. Validation Error
res_validation = query_inventory("INVALID_FORMAT_SKU")
print("\nValidation Error Result:", res_validation)

# 3. Valid Empty Result (isError: False!)
res_empty = query_inventory("SKU-99999")
print("\nValid Empty Result:", res_empty)

# 4. Business Rule Violation
res_business = issue_refund("CUST-100", 650.0)
print("\nBusiness Rule Error Result:", res_business)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Transient errors have `isRetryable == True`.
2. Validation errors have `isRetryable == False`.
3. Valid empty results have `isError == False`.
4. Business errors have `isRetryable == False` and include details.

In [ ]:
# Step 4: Verification Checks
assert res_transient["isError"] is True
assert res_transient["errorCategory"] == "transient"
assert res_transient["isRetryable"] is True

assert res_validation["isError"] is True
assert res_validation["errorCategory"] == "validation"
assert res_validation["isRetryable"] is False

assert res_empty["isError"] is False
assert res_empty["stock_count"] == 0
assert res_empty["is_empty_result"] is True

assert res_business["isError"] is True
assert res_business["errorCategory"] == "business"
assert res_business["isRetryable"] is False

print(" All verification assertions passed! MCP error taxonomy verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Always return structured errors using `isError: true` with an explicit `errorCategory`.
- **Rule 2**: Mark `isRetryable: true` ONLY for transient infrastructure failures (timeouts, rate limits).
- **Rule 3**: Never return an error for valid empty query results (0 matches). Return `isError: false` and `stock_count: 0`.